In [1]:
!pip install -U sentence-transformers datasets accelerate faiss-cpu scikit-learn pandas matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 15.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 77.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 96.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 94.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 100.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 83.3 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
  Attempting uninstall: scikit-learn
    Foun

In [2]:
import os
import torch
import pandas as pd
import numpy as np

from datasets import load_dataset

from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer
)

from sentence_transformers.sentence_transformer.training_args import (
    SentenceTransformerTrainingArguments
)

from sentence_transformers.sentence_transformer.losses import (
    MultipleNegativesRankingLoss
)

from sentence_transformers.sentence_transformer.evaluation import (
    EmbeddingSimilarityEvaluator
)

from sentence_transformers.util import cos_sim

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU Memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
else:
    print("Không có GPU, nên bật GPU trước khi train.")

Device: cuda
GPU: Tesla T4
GPU Memory: 14.56 GB


In [4]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
OUTPUT_DIR = "model_finetuned_biencoder"
FINAL_MODEL_DIR = "final_biencoder_model"

# Chọn chế độ train
# "fast": chạy nhanh để test code
# "strong": dùng cho project cuối kì
TRAIN_MODE = "strong"

if TRAIN_MODE == "fast":
    TRAIN_SIZE = 50000
    EVAL_SIZE = 2000
    EPOCHS = 1
    BATCH_SIZE = 64
else:
    TRAIN_SIZE = 200000
    EVAL_SIZE = 5000
    EPOCHS = 2
    BATCH_SIZE = 64

print("Train mode:", TRAIN_MODE)
print("Train size:", TRAIN_SIZE)
print("Eval size:", EVAL_SIZE)
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)

Train mode: strong
Train size: 200000
Eval size: 5000
Epochs: 2
Batch size: 64


In [5]:
model = SentenceTransformer(MODEL_NAME)
model.max_seq_length = 128
model.to(device)

print(model)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)


In [6]:
train_dataset = load_dataset(
    "sentence-transformers/all-nli",
    "pair",
    split="train"
)

print(train_dataset)
print(train_dataset[0])

README.md: 0.00B [00:00, ?B/s]

pair/train-00000-of-00001.parquet:   0%|          | 0.00/26.2M [00:00<?, ?B/s]

pair/dev-00000-of-00001.parquet:   0%|          | 0.00/645k [00:00<?, ?B/s]

pair/test-00000-of-00001.parquet:   0%|          | 0.00/666k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/314315 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/6808 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6831 [00:00<?, ? examples/s]

Dataset({
    features: ['anchor', 'positive'],
    num_rows: 314315
})
{'anchor': 'A person on a horse jumps over a broken down airplane.', 'positive': 'A person is outdoors, on a horse.'}


In [7]:
train_dataset = train_dataset.shuffle(seed=42)

if TRAIN_SIZE is not None:
    train_dataset = train_dataset.select(range(min(TRAIN_SIZE, len(train_dataset))))

print("Final train size:", len(train_dataset))
print(train_dataset[0])

Final train size: 200000
{'anchor': 'A child with goggles and a floaty swimming in a body of water.', 'positive': 'A child is wearing a bathing suit.'}


In [8]:
sts_eval = load_dataset(
    "sentence-transformers/stsb",
    split="validation"
)

if EVAL_SIZE is not None:
    sts_eval = sts_eval.select(range(min(EVAL_SIZE, len(sts_eval))))

print(sts_eval)
print(sts_eval[0])

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/471k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/142k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/108k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5749 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1379 [00:00<?, ? examples/s]

Dataset({
    features: ['sentence1', 'sentence2', 'score'],
    num_rows: 1500
})
{'sentence1': 'A man with a hard hat is dancing.', 'sentence2': 'A man wearing a hard hat is dancing.', 'score': 1.0}


In [9]:
dev_evaluator = EmbeddingSimilarityEvaluator(
    sentences1=sts_eval["sentence1"],
    sentences2=sts_eval["sentence2"],
    scores=[score / 5.0 for score in sts_eval["score"]],
    name="sts-dev"
)

In [10]:
print("Evaluating base model...")

base_result = dev_evaluator(model)

print("Base model result:")
print(base_result)

Evaluating base model...
Base model result:
{'sts-dev_pearson_cosine': 0.8695950729684202, 'sts-dev_spearman_cosine': 0.867097774946541}


In [11]:
train_loss = MultipleNegativesRankingLoss(model)

print(train_loss)

MultipleNegativesRankingLoss(
  (model): SentenceTransformer(
    (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
    (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
    (2): Normalize({})
  )
)


In [12]:
training_args = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    learning_rate=2e-5,
    warmup_ratio=0.1,

    fp16=True if device == "cuda" else False,

    eval_strategy="steps",
    eval_steps=1000,

    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,

    logging_steps=100,

    run_name="fine-tuned-biencoder-semantic-search"
)

The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.
Currently using DataParallel (DP) for multi-gpu training, while DistributedDataParallel (DDP) is recommended for faster training. See https://sbert.net/docs/sentence_transformer/training/distributed.html for more information.


In [13]:
trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=sts_eval,
    loss=train_loss,
    evaluator=dev_evaluator
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

In [14]:
trainer.train()

Step,Training Loss,Validation Loss,Sts-dev Pearson Cosine,Sts-dev Spearman Cosine
1000,0.215927,1.370099,0.874260,0.871017
2000,0.208118,1.393958,0.873509,0.870495
3000,0.200490,1.393690,0.873971,0.871088


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3126, training_loss=0.21293473045412578, metrics={'train_runtime': 838.0361, 'train_samples_per_second': 477.306, 'train_steps_per_second': 3.73, 'total_flos': 0.0, 'train_loss': 0.21293473045412578, 'epoch': 2.0})

In [15]:
print("Evaluating fine-tuned model...")

finetuned_result = dev_evaluator(model)

print("Fine-tuned model result:")
print(finetuned_result)

Evaluating fine-tuned model...
Fine-tuned model result:
{'sts-dev_pearson_cosine': 0.8738712616361576, 'sts-dev_spearman_cosine': 0.8709462502777147}


In [16]:
model.save(FINAL_MODEL_DIR)

print("Saved model to:", FINAL_MODEL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model to: final_biencoder_model


In [17]:
sentences = [
    "A man is playing football.",
    "A person is playing soccer.",
    "A woman is cooking in the kitchen.",
    "The stock market is going down.",
    "Students are studying in a classroom."
]

embeddings = model.encode(sentences, convert_to_tensor=True)
similarity_matrix = cos_sim(embeddings, embeddings)

df = pd.DataFrame(
    similarity_matrix.cpu().numpy(),
    index=sentences,
    columns=sentences
)

df

,A man is playing football.,A person is playing soccer.,A woman is cooking in the kitchen.,The stock market is going down.,Students are studying in a classroom.
A man is playing football.,1.000000,0.365272,-0.071143,0.053125,-0.106535
A person is playing soccer.,0.365272,1.000000,0.116141,-0.054114,-0.028049
A woman is cooking in the kitchen.,-0.071143,0.116141,1.000000,-0.043699,0.077853
The stock market is going down.,0.053125,-0.054114,-0.043699,1.000000,0.076596
Students are studying in a classroom.,-0.106535,-0.028049,0.077853,0.076596,1.000000


In [18]:
corpus = [
    "A man is playing football on the field.",
    "A woman is preparing food in the kitchen.",
    "A group of students are learning mathematics.",
    "The car is parked near the building.",
    "A person is playing soccer with friends.",
    "The weather is very cold today.",
    "A child is reading a book."
]

query = "Someone is playing football."

corpus_embeddings = model.encode(corpus, convert_to_tensor=True)
query_embedding = model.encode(query, convert_to_tensor=True)

scores = cos_sim(query_embedding, corpus_embeddings)[0]

top_k = 5
results = torch.topk(scores, k=top_k)

print("Query:", query)
print()

for score, idx in zip(results.values, results.indices):
    print("Score:", round(float(score), 4))
    print("Text:", corpus[int(idx)])
    print()

Query: Someone is playing football.

Score: 0.7342
Text: A man is playing football on the field.

Score: 0.3491
Text: A person is playing soccer with friends.

Score: 0.0981
Text: A child is reading a book.

Score: 0.0388
Text: A woman is preparing food in the kitchen.

Score: 0.0372
Text: The car is parked near the building.



In [19]:
import faiss

corpus = [
    "A man is playing football on the field.",
    "A woman is preparing food in the kitchen.",
    "A group of students are learning mathematics.",
    "The car is parked near the building.",
    "A person is playing soccer with friends.",
    "The weather is very cold today.",
    "A child is reading a book.",
    "Artificial intelligence is changing many industries.",
    "Machine learning models can learn from data.",
    "Natural language processing is a field of AI."
]

corpus_embeddings = model.encode(
    corpus,
    convert_to_numpy=True,
    normalize_embeddings=True
)

dimension = corpus_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(corpus_embeddings)

print("FAISS index size:", index.ntotal)

FAISS index size: 10


In [20]:
query = "AI and language understanding"

query_embedding = model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
)

scores, indices = index.search(query_embedding, k=5)

print("Query:", query)
print()

for score, idx in zip(scores[0], indices[0]):
    print("Score:", round(float(score), 4))
    print("Text:", corpus[idx])
    print()

Query: AI and language understanding

Score: 0.6559
Text: Natural language processing is a field of AI.

Score: 0.3863
Text: Artificial intelligence is changing many industries.

Score: 0.3555
Text: Machine learning models can learn from data.

Score: 0.2194
Text: A group of students are learning mathematics.

Score: 0.1248
Text: A child is reading a book.



In [21]:
!zip -r final_biencoder_model.zip final_biencoder_model

  adding: final_biencoder_model/ (stored 0%)
  adding: final_biencoder_model/README.md (deflated 69%)
  adding: final_biencoder_model/model.safetensors (deflated 9%)
  adding: final_biencoder_model/sentence_bert_config.json (deflated 43%)
  adding: final_biencoder_model/modules.json (deflated 64%)
  adding: final_biencoder_model/config.json (deflated 52%)
  adding: final_biencoder_model/tokenizer.json (deflated 71%)
  adding: final_biencoder_model/2_Normalize/ (stored 0%)
  adding: final_biencoder_model/config_sentence_transformers.json (deflated 40%)
  adding: final_biencoder_model/1_Pooling/ (stored 0%)
  adding: final_biencoder_model/1_Pooling/config.json (deflated 16%)
  adding: final_biencoder_model/tokenizer_config.json (deflated 45%)


In [22]:
loaded_model = SentenceTransformer(FINAL_MODEL_DIR)

test_query = "A man is playing soccer."
test_doc = "Someone is playing football."

emb1 = loaded_model.encode(test_query, convert_to_tensor=True)
emb2 = loaded_model.encode(test_doc, convert_to_tensor=True)

score = cos_sim(emb1, emb2)

print("Similarity score:", float(score))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Similarity score: 0.35492852330207825


In [23]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

model = SentenceTransformer("final_biencoder_model")

test_pairs = [
    # Cùng nghĩa
    ("A man is playing soccer.", "Someone is playing football."),
    ("A woman is cooking dinner.", "A person is preparing food."),
    ("Students are studying in class.", "Pupils are learning in a classroom."),
    ("A child is reading a book.", "A kid is reading a novel."),

    # Khác nghĩa
    ("A man is playing soccer.", "The stock market is crashing."),
    ("A woman is cooking dinner.", "An airplane is taking off."),
    ("Students are studying in class.", "A dog is sleeping on the floor."),
]

for s1, s2 in test_pairs:
    emb1 = model.encode(s1, convert_to_tensor=True)
    emb2 = model.encode(s2, convert_to_tensor=True)

    score = float(cos_sim(emb1, emb2))

    print("=" * 60)
    print("Sentence 1:", s1)
    print("Sentence 2:", s2)
    print("Similarity:", round(score, 4))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Sentence 1: A man is playing soccer.
Sentence 2: Someone is playing football.
Similarity: 0.3549
Sentence 1: A woman is cooking dinner.
Sentence 2: A person is preparing food.
Similarity: 0.4174
Sentence 1: Students are studying in class.
Sentence 2: Pupils are learning in a classroom.
Similarity: 0.8149
Sentence 1: A child is reading a book.
Sentence 2: A kid is reading a novel.
Similarity: 0.9016
Sentence 1: A man is playing soccer.
Sentence 2: The stock market is crashing.
Similarity: -0.0292
Sentence 1: A woman is cooking dinner.
Sentence 2: An airplane is taking off.
Similarity: 0.0386
Sentence 1: Students are studying in class.
Sentence 2: A dog is sleeping on the floor.
Similarity: -0.0353


In [24]:
corpus = [
    "A man is playing football.",
    "A woman is cooking dinner.",
    "Students are studying mathematics.",
    "The stock market is falling today.",
    "Artificial intelligence is transforming industry."
]

query = "Someone is playing soccer."

query_emb = model.encode(query, convert_to_tensor=True)
corpus_emb = model.encode(corpus, convert_to_tensor=True)

scores = cos_sim(query_emb, corpus_emb)[0]

for sentence, score in zip(corpus, scores):
    print(f"{float(score):.4f} -> {sentence}")

0.3603 -> A man is playing football.
0.0827 -> A woman is cooking dinner.
0.0034 -> Students are studying mathematics.
-0.0383 -> The stock market is falling today.
0.0209 -> Artificial intelligence is transforming industry.
